# Tahap 1: Gambaran Skenario CD dan Tujuan Release

CD pada mini project ini berarti menyiapkan model agar siap digunakan dalam skenario release lokal. Istilah yang digunakan adalah **Skenario CD** atau **Simulasi CD**, bukan deployment production.

Model yang sudah dilatih disimpan dalam format `.joblib`. Pada tahap ini model tidak ditraining ulang. Notebook hanya menyiapkan model registry, rencana API lokal, strategi Shadow Deployment Simulation, monitoring, rollback, dan diagram pipeline.

Tidak ada deployment cloud sungguhan pada project ini. API hanya simulasi lokal untuk kebutuhan pembelajaran dan evidence assignment.


In [ ]:
from pathlib import Path
from datetime import datetime
import json

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
elif not (project_root / "data").exists():
    project_root = project_root.parent

models_dir = project_root / "models"
reports_dir = project_root / "reports"
diagrams_dir = project_root / "diagrams"
api_dir = project_root / "api"
notebooks_dir = project_root / "notebooks"

for folder in [models_dir, reports_dir, diagrams_dir, api_dir, notebooks_dir]:
    folder.mkdir(parents=True, exist_ok=True)

model_path = models_dir / "baseline_price_model.joblib"
final_report_path = reports_dir / "final_report.md"
registry_path = models_dir / "model_registry.json"
diagram_md_path = diagrams_dir / "mlops_pipeline_vertical.md"
diagram_png_path = diagrams_dir / "mlops_pipeline_vertical.png"

print("Project root:", project_root)

In [ ]:
model_file_available = model_path.exists()
final_report_available = final_report_path.exists()
api_file_available = (api_dir / "app.py").exists()
ready_for_cd_design = all([
    model_file_available,
    final_report_available,
    api_file_available,
])

readiness_summary = pd.DataFrame({
    "Readiness Item": [
        "Model file available",
        "Final report available",
        "API skeleton available",
        "Ready for CD scenario design",
    ],
    "Status": [
        "YES" if model_file_available else "NO",
        "YES" if final_report_available else "NO",
        "YES" if api_file_available else "NO",
        "YES" if ready_for_cd_design else "NO",
    ],
    "Path / Note": [
        str(model_path),
        str(final_report_path),
        str(api_dir / "app.py"),
        "Proceed with local CD design only" if ready_for_cd_design else "Complete earlier steps first",
    ],
})

display(readiness_summary)

# Tahap 2: Desain Model Registry Sederhana

Model registry digunakan untuk mencatat versi model, metrik, lokasi file model, status approval, dan catatan monitoring.

Untuk mini project ini, registry dibuat sebagai file JSON sederhana di `models/model_registry.json`. Model dapat dianggap approved untuk Shadow Deployment Simulation karena CI PASS dan CT mostly PASS, dengan satu warning pada error service name yang perlu dimonitor.


In [3]:
model_metadata = {
    "model_name": "online_transport_fare_estimator",
    "model_version": "v1.0-baseline-random-forest",
    "model_type": "Random Forest Regressor",
    "model_file": "models/baseline_price_model.joblib",
    "target_column": "price",
    "problem_type": "regression",
    "features_used": [
        "distance",
        "cab_type",
        "source",
        "destination",
        "name",
        "hour",
        "day",
        "month",
        "day_of_week",
    ],
    "features_excluded": {
        "id": "identifier",
        "product_id": "identifier or product code",
        "price": "target column",
        "surge_multiplier": "closely related to the pricing mechanism",
    },
    "training_dataset": "data/processed/cleaned_cab_rides.csv",
    "rows_after_cleaning": 637976,
    "MAE": 1.4254,
    "RMSE": 2.6181,
    "R2_score": 0.9214,
    "CI_status": "PASS",
    "CT_status": "PASS with monitoring warning for service-name group error",
    "approval_status": "Approved for Shadow Deployment Simulation",
    "warning_notes": [
        "Highest service-name error group is Lux Black XL.",
        "Use Shadow Deployment Simulation first and monitor group-level MAE before promotion.",
        "This model is a learning simulation, not a real production pricing system.",
    ],
    "created_at": datetime.now().isoformat(timespec="seconds"),
}

registry = {"registered_models": [model_metadata]}
registry_path.write_text(json.dumps(registry, indent=2), encoding="utf-8")
print("Model registry saved to:", registry_path)
display(pd.DataFrame([model_metadata]))


Model registry saved to: d:\Tugas Kuliah\SEM 6\Proyek Data Mining\mlops-tarif-transportasi-online\models\model_registry.json


,model_name,model_version,model_type,model_file,target_column,problem_type,features_used,features_excluded,training_dataset,rows_after_cleaning,MAE,RMSE,R2_score,CI_status,CT_status,approval_status,warning_notes,created_at
0,online_transport_fare_estimator,v1.0-baseline-random-forest,Random Forest Regressor,models/baseline_price_model.joblib,price,regression,"[distance, cab_type, source, destination, name...","{'id': 'identifier', 'product_id': 'identifier...",data/processed/cleaned_cab_rides.csv,637976,1.4254,2.6181,0.9214,PASS,PASS with monitoring warning for service-name ...,Approved for Shadow Deployment Simulation,[Highest service-name error group is Lux Black...,2026-06-15T07:42:17


# Tahap 3: Rencana API dan Skeleton FastAPI Lokal

Model dapat dibungkus dalam API agar alur prediction lebih mudah disimulasikan. API pada project ini hanya simulasi lokal, bukan layanan cloud atau production deployment.

Endpoint utama menerima input data perjalanan saja: `distance`, `cab_type`, `source`, `destination`, `name`, dan `time_stamp`.

API membuat fitur waktu secara internal menggunakan preprocessing yang sama dengan training. API tidak menerima `price` karena `price` adalah target dan tidak menggunakan `surge_multiplier` pada endpoint utama.


In [4]:
app_code = """from pathlib import Path
import sys

import joblib
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel


PROJECT_ROOT = Path(__file__).resolve().parents[1]
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from preprocessing import create_time_features  # noqa: E402


MODEL_VERSION = "v1.0-baseline-random-forest"
MODEL_PATH = PROJECT_ROOT / "models" / "baseline_price_model.joblib"
FEATURE_COLUMNS = [
    "distance",
    "cab_type",
    "source",
    "destination",
    "name",
    "hour",
    "day",
    "month",
    "day_of_week",
]


app = FastAPI(
    title="Online Transportation Fare Estimation API",
    description="Local learning simulation API for trip-data-only fare estimation in an MLOps mini project.",
    version=MODEL_VERSION,
)

model = joblib.load(MODEL_PATH)


class FarePredictionRequest(BaseModel):
    distance: float
    cab_type: str
    source: str
    destination: str
    name: str
    time_stamp: int


def validate_request(data):
    if data.distance <= 0:
        raise HTTPException(status_code=400, detail="distance must be greater than 0.")

    text_fields = {
        "cab_type": data.cab_type,
        "source": data.source,
        "destination": data.destination,
        "name": data.name,
    }
    for field_name, value in text_fields.items():
        if not value or not value.strip():
            raise HTTPException(status_code=400, detail=f"{field_name} must not be empty.")

    converted_time = pd.to_datetime(data.time_stamp, unit="ms", errors="coerce")
    if pd.isna(converted_time):
        raise HTTPException(status_code=400, detail="time_stamp must be a valid millisecond timestamp.")


@app.get("/")
def read_root():
    return {
        "message": "Online transportation fare estimation API is running.",
        "model_version": MODEL_VERSION,
        "note": "Local learning simulation using trip data only; not a real production pricing system.",
    }


@app.post("/predict")
def predict_price(request: FarePredictionRequest):
    validate_request(request)

    if hasattr(request, "model_dump"):
        request_data = request.model_dump()
    else:
        request_data = request.dict()

    input_df = pd.DataFrame([request_data])
    input_df = create_time_features(input_df)
    prediction_input = input_df[FEATURE_COLUMNS]
    estimated_price = float(model.predict(prediction_input)[0])

    return {
        "estimated_price": round(estimated_price, 2),
        "model_version": MODEL_VERSION,
        "note": "Local learning simulation using trip data only; price and surge_multiplier are not prediction inputs.",
    }
"""

request_example = {
    "distance": 2.5,
    "cab_type": "Uber",
    "source": "Back Bay",
    "destination": "North End",
    "name": "UberX",
    "time_stamp": 1544952607890,
}

api_readme = """# API Simulasi Lokal Estimasi Tarif

Folder ini berisi skeleton FastAPI untuk mini project MLOps. API ini hanya simulasi lokal untuk pembelajaran, bukan sistem pricing produksi nyata dan bukan deployment cloud.

API menggunakan data perjalanan saja untuk membuat estimasi tarif.

## Input Endpoint `/predict`

API menggunakan data perjalanan saja:

- `distance`
- `cab_type`
- `source`
- `destination`
- `name`
- `time_stamp`

API tidak menerima `price` sebagai input karena `price` adalah target prediksi. API juga tidak menggunakan `surge_multiplier` pada endpoint utama.

## Menjalankan API Lokal

Install dependencies:

```bash
pip install fastapi uvicorn joblib pandas scikit-learn
```

Jalankan dari root project:

```bash
uvicorn api.app:app --reload
```

Endpoint utama:

```text
POST http://127.0.0.1:8000/predict
```

Contoh request tersedia di `api/request_example.json`.

## Batasan

- API hanya simulasi lokal.
- Tidak ada deployment cloud sungguhan.
- API tidak boleh digunakan untuk menentukan tarif nyata.
"""

(api_dir / "app.py").write_text(app_code, encoding="utf-8")
(api_dir / "request_example.json").write_text(json.dumps(request_example, indent=2), encoding="utf-8")
(api_dir / "README.md").write_text(api_readme, encoding="utf-8")

api_summary = pd.DataFrame({
    "API File": ["api/app.py", "api/request_example.json", "api/README.md"],
    "Created": [
        (api_dir / "app.py").exists(),
        (api_dir / "request_example.json").exists(),
        (api_dir / "README.md").exists(),
    ],
    "Purpose": [
        "Local FastAPI skeleton",
        "Example request body",
        "Local run instructions",
    ],
})

display(api_summary)


,API File,Created,Purpose
0,api/app.py,True,Local FastAPI skeleton
1,api/request_example.json,True,Example request body
2,api/README.md,True,Local run instructions


# Tahap 4: Skenario Shadow Deployment Simulation

Strategi yang dipilih adalah **Shadow Deployment Simulation**. Pada project ini, strategi tersebut masih berupa rancangan/simulasi, bukan production deployment.

Model baru dapat menghasilkan prediksi di background terlebih dahulu, hasil prediksi dapat dibandingkan dengan baseline, dan error dapat dimonitor sebelum model dipromosikan dalam simulasi. Jika monitoring tidak stabil, rollback dapat dilakukan ke model approved sebelumnya.


In [5]:
cd_checklist = pd.DataFrame([
    {"CD Component": "Model format", "Design Choice": ".joblib", "Purpose": "Store the trained sklearn pipeline", "Status": "READY" if model_file_available else "MISSING", "Notes": "Saved as models/baseline_price_model.joblib"},
    {"CD Component": "Model registry", "Design Choice": "models/model_registry.json", "Purpose": "Track model version, metrics, approval, and warnings", "Status": "READY" if registry_path.exists() else "MISSING", "Notes": "Simple JSON registry for mini project"},
    {"CD Component": "API framework", "Design Choice": "FastAPI", "Purpose": "Expose local prediction endpoint", "Status": "PLANNED", "Notes": "Local demo only, not cloud deployment"},
    {"CD Component": "Prediction endpoint", "Design Choice": "/predict", "Purpose": "Return estimated fare from trip input", "Status": "READY", "Notes": "Trip data only; no price or surge_multiplier input"},
    {"CD Component": "Deployment strategy", "Design Choice": "Shadow Deployment Simulation", "Purpose": "Test predictions in background before promotion", "Status": "DESIGNED", "Notes": "Simulation strategy for this assignment"},
    {"CD Component": "Rollback strategy", "Design Choice": "Previous approved model version", "Purpose": "Return to earlier model if monitoring is unstable", "Status": "DESIGNED", "Notes": "Current project has one registered baseline version"},
    {"CD Component": "Monitoring metrics", "Design Choice": "MAE, RMSE, R2, group-level MAE", "Purpose": "Check accuracy and stability after release simulation", "Status": "DESIGNED", "Notes": "Service-name error warning must be monitored"},
    {"CD Component": "Logging plan", "Design Choice": "Input, output, timestamp, model version", "Purpose": "Support monitoring review and debugging", "Status": "PLANNED", "Notes": "No logging service is created in this step"},
    {"CD Component": "Safety note", "Design Choice": "Learning simulation only", "Purpose": "Avoid claiming real production pricing use", "Status": "DOCUMENTED", "Notes": "Not a real pricing system"},
])

display(cd_checklist)


,CD Component,Design Choice,Purpose,Status,Notes
0,Model format,.joblib,Store the trained sklearn pipeline,READY,Saved as models/baseline_price_model.joblib
1,Model registry,models/model_registry.json,"Track model version, metrics, approval, and wa...",READY,Simple JSON registry for mini project
2,API framework,FastAPI,Expose local prediction endpoint,PLANNED,"Local demo only, not cloud deployment"
3,Prediction endpoint,/predict,Return estimated fare from trip input,READY,Trip data only; no price or surge_multiplier i...
4,Deployment strategy,Shadow Deployment Simulation,Test predictions in background before promotion,DESIGNED,Simulation strategy for this assignment
5,Rollback strategy,Previous approved model version,Return to earlier model if monitoring is unstable,DESIGNED,Current project has one registered baseline ve...
6,Monitoring metrics,"MAE, RMSE, R2, group-level MAE",Check accuracy and stability after release sim...,DESIGNED,Service-name error warning must be monitored
7,Logging plan,"Input, output, timestamp, model version",Support monitoring review and debugging,PLANNED,No logging service is created in this step
8,Safety note,Learning simulation only,Avoid claiming real production pricing use,DOCUMENTED,Not a real pricing system


## Catatan Skenario CD

Tabel di atas menjadi bukti rancangan CD untuk assignment. Komponen utamanya adalah model `.joblib`, model registry, FastAPI lokal, endpoint `/predict`, Shadow Deployment Simulation, monitoring, dan rollback strategy.

Report CD terpisah diperbarui di folder `reports/`, sedangkan notebook ini difokuskan pada proses desain dan bukti teknis.


# Tahap 5: Diagram Mini Pipeline MLOps

Diagram pipeline menunjukkan alur vertikal dari raw trip data sampai monitoring dan keputusan rollback atau promotion. Diagram ini menjadi salah satu output utama assignment karena memperlihatkan hubungan CI, CT, dan CD secara ringkas.

In [ ]:
mermaid_diagram = """```mermaid
flowchart TD
    A["1. Raw Trip Data<br/>cab_rides.csv"]
    B["2. Data Audit<br/>missing value, duplicate, validasi kolom"]
    C["3. CI Data Validation<br/>required columns, price, distance, timestamp"]
    D["4. Data Cleaning<br/>drop missing price, validasi nilai numerik"]
    E["5. Feature Engineering<br/>hour, day, month, day_of_week"]
    F["6. Preprocessing Pipeline<br/>numeric scaling, categorical encoding"]
    G["7. Model Training<br/>Dummy, Ridge, Random Forest"]
    H["8. Model Evaluation<br/>MAE, RMSE, R2 Score"]
    I["9. CT Quality Checklist<br/>performance, fairness, robustness"]
    J["10. Model Registry<br/>model_registry.json"]
    K["11. API Service<br/>FastAPI /predict"]
    L["12. Shadow Deployment Simulation<br/>local simulation only"]
    M["13. Monitoring<br/>MAE, RMSE, R2, group-level MAE"]
    N["14. Rollback / Promotion Decision<br/>approve or revert model version"]

    A --> B --> C --> D --> E --> F --> G --> H --> I --> J --> K --> L --> M --> N
```
"""

diagram_md_content = f"""# Diagram Pipeline MLOps Vertikal

{mermaid_diagram}
"""

diagram_md_path.write_text(diagram_md_content, encoding="utf-8")
print("Mermaid diagram saved to:", diagram_md_path)

In [ ]:
def draw_pipeline_png(output_path):
    nodes = [
        ("1. Raw Trip Data", "cab_rides.csv"),
        ("2. Data Audit", "missing value, duplicate, validasi kolom"),
        ("3. CI Data Validation", "required columns, price, distance, timestamp"),
        ("4. Data Cleaning", "drop missing price, validasi nilai numerik"),
        ("5. Feature Engineering", "hour, day, month, day_of_week"),
        ("6. Preprocessing Pipeline", "numeric scaling, categorical encoding"),
        ("7. Model Training", "Dummy, Ridge, Random Forest"),
        ("8. Model Evaluation", "MAE, RMSE, R2 Score"),
        ("9. CT Quality Checklist", "performance, fairness, robustness"),
        ("10. Model Registry", "model_registry.json"),
        ("11. API Service", "FastAPI /predict"),
        ("12. Shadow Deployment Simulation", "local simulation only"),
        ("13. Monitoring", "MAE, RMSE, R2, group-level MAE"),
        ("14. Rollback / Promotion Decision", "approve or revert model version"),
    ]

    fig, ax = plt.subplots(figsize=(9, 18))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 17)
    ax.axis("off")

    ax.text(
        5,
        16.35,
        "Mini Pipeline MLOps Estimasi Tarif Transportasi Online",
        ha="center",
        va="center",
        fontsize=18,
        fontweight="bold",
        color="#243447",
    )

    box_x = 1.15
    box_w = 7.7
    box_h = 0.72
    start_y = 15.05
    step_y = 1.02
    colors = ["#E8F4FA", "#EAF7EA", "#FFF4D8", "#F4EAFE"]
    edge_color = "#3D4B59"

    centers = []
    for index, (title, detail) in enumerate(nodes):
        y = start_y - index * step_y
        face_color = colors[index // 4] if index // 4 < len(colors) else "#FDECEC"
        box = FancyBboxPatch(
            (box_x, y - box_h / 2),
            box_w,
            box_h,
            boxstyle="round,pad=0.08,rounding_size=0.08",
            facecolor=face_color,
            edgecolor=edge_color,
            linewidth=1.2,
        )
        ax.add_patch(box)
        ax.text(box_x + 0.28, y + 0.12, title, ha="left", va="center", fontsize=11.5, fontweight="bold", color="#1F2A33")
        ax.text(box_x + 0.28, y - 0.16, detail, ha="left", va="center", fontsize=9.5, color="#38444D")
        centers.append((5, y))

    for (_, y1), (_, y2) in zip(centers[:-1], centers[1:]):
        ax.annotate(
            "",
            xy=(5, y2 + box_h / 2 + 0.05),
            xytext=(5, y1 - box_h / 2 - 0.05),
            arrowprops=dict(arrowstyle="->", color=edge_color, linewidth=1.2),
        )

    fig.tight_layout()
    fig.savefig(output_path, dpi=220, bbox_inches="tight")
    plt.close(fig)

try:
    draw_pipeline_png(diagram_png_path)
    png_status = "created"
except Exception as exc:
    png_status = f"not created: {exc}"

print("PNG diagram status:", png_status)
print("PNG diagram path:", diagram_png_path)

## Catatan Diagram Pipeline

Diagram menggunakan layout vertikal agar mudah dimasukkan ke laporan akhir. Node awal adalah `cab_rides.csv`, lalu alur berlanjut ke data audit, CI, preprocessing, training, evaluasi, CT, registry, API lokal, Shadow Deployment Simulation, monitoring, dan keputusan rollback atau promotion.

# Ringkasan Tahap 4

- Model sudah memiliki registry di `models/model_registry.json`.
- API lokal sudah dirancang menggunakan FastAPI dengan endpoint `/predict`.
- Strategi Shadow Deployment Simulation dipilih sebagai rancangan CD.
- Diagram pipeline vertikal sudah dibuat sebagai `diagrams/mlops_pipeline_vertical.png`.
- Skenario CD sudah memenuhi kebutuhan tugas.
- Tidak ada deployment cloud sungguhan pada tahap ini.